# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện

**Track 4 · Ngày 1 · VinUniversity AICB 2026** · Bài toán: phân loại 7 loại rừng trên
Forest CoverType bằng MLP tự viết.

Notebook này chạy từ đầu đến cuối (`Restart & Run All`). Toàn bộ code nằm trong thư mục
`code/` (`data.py`, `model.py`, `optimizer.py`, `train.py`, `plots.py`, `results_table.py`),
mọi ô đều gọi hàm trong các module đó — không có logic huấn luyện nào nằm trực tiếp trong notebook.

**Cách chạy trên Kaggle**

1. Tạo notebook mới, bật **Accelerator: GPU T4 x2** và **Settings → Internet: On**.
2. Kéo thả `code/lab.ipynb` vào Kaggle.
3. Bấm **Restart & Run All**. Ô đầu tiên sẽ **tự `git clone`** repo vào `/kaggle/working/repo`
   (đổi `REPO_URL` ở đầu ô đó nếu bạn dùng fork khác), rồi đọc `data/covtype.csv.gz` +
   `data/split_metadata.csv`.
4. Kết quả ghi vào `/kaggle/working/{figures,results}/`; ô cuối (Part 5) gom thành
   `/kaggle/working/repo/submission_<MSSV>/` đúng cấu trúc README mục 6.1.

> **Vì sao clone phải nằm trong notebook?** "Save Version → Save & Run All" của Kaggle chạy ở
> session MỚI, nên file đã clone trong session tương tác sẽ **biến mất** — đó chính là lý do
> `REPO_ROOT = None` dù đã clone. Vì vậy ô setup tự clone.
> Nếu không bật Internet: upload repo làm Kaggle Dataset, rồi sửa `REPO_URL`/`REPO_LOCAL` ở
> đầu ô setup thành lệnh `cp` từ `/kaggle/input/<tên-dataset>`.

**Quy tắc bắt buộc của lab** (đã cài trong code, không vi phạm ở đâu)

- Tập `eval` **chỉ** dùng ở Part 4, cho baseline và cấu hình cuối cùng. Mọi lựa chọn dùng `val`.
- Mỗi thí nghiệm chỉ đổi **một** yếu tố so với baseline (trừ cấu hình cuối cùng, ghi rõ trong bảng).
- Cùng một phép tách val (20%, phân tầng, seed 42), cùng số epoch cho các thí nghiệm so sánh.
- Kết luận "A tốt hơn B" phải so với ngưỡng nhiễu 2σ đo từ các seed baseline.

**Chế độ kiểm tra nhanh:** đặt `FAST = True` trong ô setup (biến môi trường `LAB_FAST=1`) để
chạy thử với 2 epoch. Chạy thật thì để `FAST = False`.

In [ ]:
# ===== 0. Môi trường, đường dẫn, cờ chạy =====
import os, sys, json, time, subprocess, shutil, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import Image, display

CWD = os.getcwd()          # thư mục chứa lab.ipynb và các module .py

def is_repo_root(p):
    p = str(p)
    return (os.path.isfile(os.path.join(p, "data", "covtype.csv.gz"))
            and os.path.isfile(os.path.join(p, "scripts", "split_data.py")))

def is_code_dir(p):
    """Thư mục chứa các module .py của bạn (tức là nơi lab.ipynb nằm cùng chúng)."""
    p = str(p)
    return all(os.path.isfile(os.path.join(p, f)) for f in ("data.py", "model.py", "train.py"))

def find_repo_root():
    """Tự dò thư mục gốc repo (thư mục chứa data/covtype.csv.gz và scripts/split_data.py).

    Đi lên tối đa 8 cấp từ thư mục chứa notebook, rồi thử các đường dẫn thường gặp
    trên Kaggle/Colab. Nhờ vậy chạy được cả khi notebook đặt ở <submission>/code/ lẫn
    ở /kaggle/working.
    """
    cands = []
    p = os.path.abspath(CWD)
    for _ in range(8):
        cands.append(p)
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    cands += [os.path.join(CWD, "K4-Track4-Day1-Neuralnetwork"),
              "/kaggle/working/repo", "/kaggle/working/K4-Track4-Day1-Neuralnetwork",
              "/content/K4-Track4-Day1-Neuralnetwork"]
    for c in cands:
        if is_repo_root(c):
            return os.path.abspath(c)
    return None

REPO_URL = "https://github.com/pvksssss/K4-Track4-Day1-Neuralnetwork.git"
REPO_LOCAL = "/kaggle/working/repo"

REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    # Tự clone. Cần bật Internet (Settings → Internet: On).
    # Lý do phải clone TRONG notebook: "Save Version → Save & Run All" của Kaggle chạy
    # session MỚI, nên mọi file đã clone ở session tương tác trước đó đều biến mất.
    print("chưa thấy repo -> đang git clone (cần Internet On)...")
    try:
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_LOCAL],
                           capture_output=True, text=True, timeout=600)
        print(r.stdout.strip() or r.stderr.strip())
        REPO_ROOT = find_repo_root()
    except Exception as e:
        print(f"git clone thất bại: {e}")

assert REPO_ROOT, (
    "không tìm thấy thư mục gốc repo. Cách xử lý:\n"
    "  1) Bật Internet: Settings → Internet: On, rồi chạy lại ô này; hoặc\n"
    "  2) chạy thủ công ở ô phía trên:\n"
    "     !git clone --depth 1 " + REPO_URL + " " + REPO_LOCAL + "\n"
    "  3) nếu dùng Kaggle Dataset: sửa REPO_URL/REPO_LOCAL ở trên cho khớp.")

# Thư mục chứa các module .py của bạn. Nếu lab.ipynb nằm cùng chỗ với .py thì dùng luôn CWD,
# còn trên Kaggle (notebook ở /kaggle/working, code ở /kaggle/working/repo/code) thì lấy từ repo.
CODE_DIR = CWD if is_code_dir(CWD) else os.path.join(REPO_ROOT, "code")
assert is_code_dir(CODE_DIR), f"không thấy data.py/model.py/train.py trong {CODE_DIR}"
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# --- Thư mục nộp (nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv):
#     * lab.ipynb nằm cùng .py trong <submission>/code/  -> OUT_DIR = <submission>
#     * lab.ipynb nằm chung chỗ với .py ở /kaggle/working -> OUT_DIR = /kaggle/working
#     * chạy trong code/ của repo (không phải thư mục nộp) -> tạo <repo>/submission_<tên>
# Có thể ép bằng biến môi trường LAB_OUT_DIR / LAB_SUBMISSION.
SUBMISSION_NAME = os.environ.get("LAB_SUBMISSION", "submission")

def pick_out_dir(cwd):
    if not is_code_dir(cwd):
        return cwd                                    # Kaggle: lab.ipynb ở /kaggle/working
    parent = os.path.dirname(cwd)
    if not parent or not os.access(parent, os.W_OK):
        return cwd                                    # thư mục cha không ghi được -> dùng cwd
    if not is_repo_root(parent):
        return parent                                 # <submission>/code/ -> <submission>
    return os.path.join(REPO_ROOT, SUBMISSION_NAME)   # đang chạy trong code/ của repo

OUT_DIR = os.environ.get("LAB_OUT_DIR") or pick_out_dir(CWD)
FIGS_DIR, RES_DIR = f"{OUT_DIR}/figures", f"{OUT_DIR}/results"
for d in (FIGS_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

# --- encoding cho subprocess (scripts/*.py in tiếng Việt; console Windows mặc định cp1252)
SUBENV = {**os.environ, "PYTHONIOENCODING": "utf-8"}

# --- cờ chạy
FAST = os.environ.get("LAB_FAST", "0") == "1"          # True = kiểm tra nhanh (2 epoch)
EPOCHS       = 2 if FAST else 20                       # số epoch chuẩn của mọi thí nghiệm
EPOCHS_LONG  = 3 if FAST else 60                       # thí nghiệm "huấn luyện lâu hơn"
EPOCHS_FINAL = 2 if FAST else 40                       # cấu hình cuối cùng
SKIP_DONE = True     # bỏ qua thí nghiệm đã có results/<exp_id>.json (tiết kiệm thời gian khi chạy lại)

# --- thiết bị
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"notebook  = {CWD}   (có các file .py ở đây: {is_code_dir(CWD)})")
print(f"CODE_DIR  = {CODE_DIR}   (nơi import data / model / optimizer / train / plots / results_table)")
print(f"OUT_DIR   = {OUT_DIR}   (figures/ results/ experiments.xlsx predictions_eval.csv đều ở đây)")
print(f"torch     = {torch.__version__}")
print(f"device    = {device}", end="")
if device == "cuda":
    print(f"  |  GPU: {torch.cuda.get_device_name(0)}")
    print(f"             bf16 phần cứng được hỗ trợ: {torch.cuda.is_bf16_supported()}")
else:
    print()
print(f"FAST={FAST}  epochs={EPOCHS}  long={EPOCHS_LONG}  final={EPOCHS_FINAL}")

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

# ---- các module của bạn
from data import prepare_data, N_NUMERIC
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, build_scheduler, clip_gradients
from train import (DEFAULT_CFG, set_seed, evaluate, predict, compute_loss, run_experiment,
                   final_eval, write_predictions, macro_f1_from_confusion, LN_C)
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx, noise_stats
print("import OK:", sorted(f.split('/')[-1] for f in
      ["data.py", "model.py", "optimizer.py", "train.py", "plots.py", "results_table.py"]))

---
## Part 0 — Dữ liệu

Chia sẵn bằng metadata: `scripts/split_data.py` đọc `data/covtype.csv.gz` + `data/split_metadata.csv`
rồi tạo `data/processed/{train,eval}.npz`. **Không** sửa metadata, **không** tự chia lại train/eval.

Sau đó: tách 20% validation từ `train` (phân tầng theo nhãn, seed 42), chuẩn hoá 10 cột số liên tục
bằng thống kê **chỉ của phần train còn lại**, rồi đưa toàn bộ lên GPU một lần (không dùng DataLoader).

In [ ]:
# ---- Chạy scripts/split_data.py (bỏ qua nếu đã có .npz) ----
need_split = not all(os.path.isfile(f"{REPO_ROOT}/data/processed/{s}.npz") for s in ("train", "eval"))
if need_split:
    subprocess.run([sys.executable, os.path.join(REPO_ROOT, "scripts", "split_data.py")],
                   cwd=REPO_ROOT, check=True, env=SUBENV)
else:
    print("data/processed/*.npz đã tồn tại -> bỏ qua split_data.py")

# ---- Nạp + tách val + chuẩn hoá + đưa lên device ----
data = prepare_data(device, val_fraction=0.2, seed=42,
                    processed_dir=os.path.join(REPO_ROOT, "data", "processed"))

In [ ]:
# ---- Tự kiểm tra Part 0 ----
cnt = {k: torch.bincount(data[f"y_{k}"], minlength=7).cpu().numpy() for k in ("tr", "val", "eval")}
dist = pd.DataFrame({k: 100 * v / v.sum() for k, v in cnt.items()}).round(2)
dist.index.name = "lớp y"
print("Tỉ lệ mẫu theo lớp (%):")
display(dist)

maj = int(data["majority_class"])
print(f"lớp đa số = {maj}")
for k in ("tr", "val", "eval"):
    acc = float((data[f"y_{k}"] == maj).float().mean())
    print(f"  'luôn đoán lớp {maj}' trên {k:4s}: accuracy = {acc:.4f}")
print("\nKỳ vọng theo GUIDE: train 464809 -> val 92962 / train 371847; eval 116203; acc đa số ≈ 0.4876")

# Chuẩn hoá: 10 cột đầu của X_tr có mean ~ 0 và std ~ 1; 44 cột nhị phân giữ nguyên 0/1
mu = data["X_tr"][:, :N_NUMERIC].mean(0).cpu().numpy()
sd = data["X_tr"][:, :N_NUMERIC].std(0, unbiased=False).cpu().numpy()
print("\nThống kê 10 cột số trên X_tr (sau chuẩn hoá):")
display(pd.DataFrame({"mean": mu.round(6), "std": sd.round(6),
                      "mean(X_val)": data["X_val"][:, :N_NUMERIC].mean(0).cpu().numpy().round(4),
                      "mean(X_eval)": data["X_eval"][:, :N_NUMERIC].mean(0).cpu().numpy().round(4)}))
print("Ghi chú: mean(X_val)/mean(X_eval) lệch 0 so với 0 vì chuẩn hoá dùng thống kê của X_tr — "
      "đây là rò rỉ thông tin nếu ta lấy thống kê từ val/eval.")

---
## Part 1 — Model và các phép thử "sức khoẻ" ban đầu

Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`,
softmax **không** nằm trong model, dropout chỉ sau ReLU của lớp ẩn.

Các phép thử rẻ (Chương 5 slide) phải làm **trước** khi huấn luyện lâu:

1. Shape và số tham số đúng quy định.
2. Loss bước 0 ≈ `ln 7 = 1.9459` (kỳ vọng khi phân bố logit cân bằng).
3. Quá khớp được một lô 20 mẫu: loss phải về gần 0.
4. Sau một lần `backward()`, **mọi** tham số phải có gradient khác `None` và khác 0.

In [ ]:
# ---- 1a/1b: shape và số tham số ----
print("Số tham số quy định (README mục 3):")
for h, n in EXPECTED_PARAMS.items():
    m = MLP(hidden=h, init="he")
    print(f"  {'->'.join(map(str, (54,) + h + (7,))):>18s}  đếm được {count_params(m):>7d}  "
          f"quy định {n:>7d}  {'OK' if count_params(m) == n else 'SAI'}")
    assert count_params(m) == n

model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)]

print("\nShape sau từng lớp (lô ngẫu nhiên 8 x 54):")
h = torch.randn(8, N_NUMERIC + 44, device=device)
with torch.no_grad():
    for m_ in model.net:
        h = m_(h)
        print(f"  {type(m_).__name__:12s} -> {tuple(h.shape)}")
assert h.shape == (8, 7)
print("logits dtype:", h.dtype, "| không có softmax trong model: OK")

# dropout đặt đúng chỗ: sau ReLU của lớp ẩn, không có trên đầu vào hay trên logit
md_ = MLP(hidden=(256, 128), dropout=0.3)
print("\nThứ tự lớp khi bật dropout:", [type(x).__name__ for x in md_.net])

In [ ]:
# ---- 1c: loss bước 0 (kỳ vọng ≈ ln 7) cho TỪNG kiểu khởi tạo ----
print(f"{'init':>8s} {'step0 val loss':>15s} {'- ln7':>9s} | std kích hoạt sau ReLU1 / ReLU2 / logits")
rows = []
for init in ("he", "default", "xavier", "normal", "zeros"):
    set_seed(1)
    m = MLP((256, 128), init=init).to(device)
    r = evaluate(m, data["X_val"], data["y_val"])
    stds = activation_stats(m, data["X_val"][:20_000])
    rows.append(dict(init=init, step0_loss=round(r["loss"], 4), gap_vs_ln7=round(r["loss"] - LN_C, 4),
                     step0_acc=round(r["acc"], 4), step0_macro_f1=round(r["macro_f1"], 4),
                     **{f"std{i+1}": round(s, 4) for i, s in enumerate(stds)}))
display(pd.DataFrame(rows))
print(f"ln 7 = {LN_C:.4f}")

In [ ]:
# ---- 1c (tiếp): quá khớp 20 mẫu — loss phải về gần 0 ----
def overfit_test(n_samples=20, steps=400, lr=0.1, seed=1):
    """Quá khớp một lô nhỏ: bằng chứng model + loss + optimizer là đúng."""
    set_seed(seed)
    m = MLP((256, 128), dropout=0.0, init="he").to(device)
    opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9)
    xb, yb = data["X_tr"][:n_samples], data["y_tr"][:n_samples]
    losses, accs = [], []
    for step in range(1, steps + 1):
        m.train()
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(m(xb), yb)
        loss.backward()
        opt.step()
        losses.append(loss.detach().item())
        if step % 20 == 0 or step == 1:
            accs.append(float((m(xb).argmax(1) == yb).float().mean()))
    return losses, accs

losses, accs = overfit_test()
print(f"quá khớp 20 mẫu (SGD+mom lr=0.1, 400 bước): loss {losses[0]:.4f} -> {losses[-1]:.6f}")
print("accuracy trên 20 mẫu đó theo thời điểm:", [round(a, 3) for a in accs])
assert losses[-1] < 1e-2, "quá khớp 20 mẫu chưa về gần 0 -> gần như chắc chắn lỗi code"

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(losses, lw=1.2); ax[0].set_yscale("log")
ax[0].set_xlabel("bước"); ax[0].set_ylabel("loss trên 20 mẫu"); ax[0].set_title("Quá khớp 20 mẫu")
ax[0].axhline(losses[-1], ls="--", c="gray", lw=1, label=f"loss cuối = {losses[-1]:.2e}")
ax[0].legend(fontsize=8)
ax[1].step(range(20, 20 * len(accs) + 1, 20), accs, where="post", marker="o", ms=3)
ax[1].set_xlabel("bước"); ax[1].set_ylabel("accuracy"); ax[1].set_ylim(-0.05, 1.05)
ax[1].set_title("Accuracy trên 20 mẫu"); ax[1].axhline(1.0, ls="--", c="gray", lw=1)
plt.tight_layout(); plt.show()

In [ ]:
# ---- 1d: gradient có chảy tới MỌI tham số không ----
set_seed(1)
m = MLP((256, 128), init="he").to(device)
opt = torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9)
xb, yb = data["X_tr"][:512], data["y_tr"][:512]
opt.zero_grad(set_to_none=True)
F.cross_entropy(m(xb), yb).backward()

print(f"{'tham số':>12s} {'shape':>12s} {'|grad| L2':>12s}  khác None  khác 0")
ok = True
for name, p in m.named_parameters():
    g = p.grad
    has = g is not None
    nrm = float(g.norm()) if has else float("nan")
    ok &= has and nrm > 0
    print(f"{name:>12s} {str(tuple(p.shape)):>12s} {nrm:12.6f}  {str(has):>9s}  {str(has and nrm > 0):>7s}")
assert ok, "có tham số không nhận được gradient khác 0"
print("\nOK: gradient chảy tới tất cả", sum(1 for _ in m.parameters()), "nhóm tham số.")

**Nhận xét Part 1 (viết lại bằng lời của bạn, chèn số đo ở trên):**

- Số tham số / shape logits: ...
- Loss bước 0 = ... so với `ln 7 = 1.9459`. **Chú ý:** loss bước 0 *không* bằng `ln 7` ngay cả
  khi mô hình "đúng", vì nó phụ thuộc phân bố phương sai của logit lớp cuối. Bảng ở trên cho thấy
  điều đó: khởi tạo He làm logit có phương sai lớn hơn (nên loss cao hơn `ln 7`), còn khởi tạo
  mặc định của `nn.Linear` cho logit nhỏ (loss gần `ln 7`). Vậy tiêu chí đúng là
  **loss bước 0 không quá xa `ln 7` và không phụ thuộc dữ liệu/lớp đa số**; nếu lệch rất xa
  (ví dụ > 0.5) thì kiểm tra lại tỉ lệ `W` lớp cuối, bias, và việc chuẩn hoá đầu vào.
- Quá khớp 20 mẫu: loss ... ; nếu không về gần 0 thì lỗi nằm ở code chứ không phải năng lực mô hình.

---
## Part 2 — Pipeline và baseline

`run_experiment(cfg, data)` là **một hàm duy nhất** cho mọi thí nghiệm: nhận dict cấu hình,
trả về lịch sử mỗi epoch + tóm tắt + `best_state` (epoch có `val_loss` thấp nhất).

Baseline (theo GUIDE): `M-base`, khởi tạo He, cross-entropy, **SGD + momentum 0.9**,
batch 512, **20 epoch**, dropout 0, không clip, FP32. `lr` chọn bằng `val`.

Mỗi epoch ghi: `train_loss` (**đo ở chế độ `eval()`** trên một tập con cố định 50 000 mẫu để
đường train và val cùng thang đo), `val_loss`, `val_acc`, `val_macro_f1`,
`grad_norm` (trung bình / min / max trong epoch, **đo trước khi clip**), thời gian epoch, lr hiện hành.

In [ ]:
# ---- Bọc run_experiment: lưu JSON + vẽ ảnh, và bỏ qua nếu đã có kết quả ----
RESULTS = {}

def show_fig(exp_id):
    p = f"{FIGS_DIR}/{exp_id}.png"
    if os.path.isfile(p):
        display(Image(p))

def run_exp(cfg, plot=True):
    """Chạy một thí nghiệm = đổi dict cfg. Lưu results/<exp_id>.json + figures/<exp_id>.png."""
    cfg = {**DEFAULT_CFG, **cfg}
    cfg["hidden"] = tuple(cfg["hidden"])
    cfg["epochs"] = int(cfg["epochs"])
    eid = cfg["exp_id"]
    p = f"{RES_DIR}/{eid}.json"
    if SKIP_DONE and os.path.isfile(p):
        with open(p, encoding="utf-8") as f:
            old = json.load(f)
        old_cfg = {**old["cfg"], "hidden": tuple(old["cfg"]["hidden"])}
        if old_cfg == cfg:
            print(f"  [{eid}] đã có trong results/ và cấu hình không đổi -> bỏ qua")
            RESULTS[eid] = {**old, "best_state": None}
            return RESULTS[eid]
        print(f"  [{eid}] có kết quả cũ nhưng cấu hình đã đổi -> chạy lại")
    res = run_experiment(cfg, data)
    save_result(res, RES_DIR)
    if plot:
        plot_run(res, f"{FIGS_DIR}/{eid}.png")
    RESULTS[eid] = res
    return res

def brief(exp_ids):
    """Bảng tóm tắt nhiều thí nghiệm + chênh lệch so với trung bình baseline và ngưỡng 2σ."""
    base_mean = NOISE["val_macro_f1"]["mean"]
    two_sig = NOISE["val_macro_f1"]["two_sigma"]
    recs = []
    for eid in exp_ids:
        s = RESULTS[eid]["summary"]
        d = s["val_macro_f1"] - base_mean
        recs.append(dict(exp_id=eid, val_macro_f1=round(s["val_macro_f1"], 4),
                         delta_vs_base=round(d, 4),
                         vượt_2σ="Có" if abs(d) > two_sig else "Không",
                         val_acc=round(s["val_acc"], 4),
                         best_val_loss=round(s["best_val_loss"], 4), best_epoch=s["best_epoch"],
                         gap_val_minus_train=round(s["final_val_loss"] - s["final_train_loss"], 4),
                         s_per_epoch=round(s["time_per_epoch_s"], 2),
                         peak_MB=round(s["peak_mem_MB"], 0),
                         diverged="Y" if s["diverged"] else "N"))
    return pd.DataFrame(recs).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)

BASE = dict(exp_id="base-s1", group="baseline",
            description=f"Baseline M-base (SGD+mom, He, {EPOCHS} ep)",
            loss="ce", optimizer="sgd_momentum", lr=0.05, weight_decay=0.0, momentum=0.9,
            batch=512, epochs=EPOCHS, hidden=(256, 128), dropout=0.0, init="he",
            clip_norm=None, precision="fp32", scheduler=None, seed=1, notes="")
print("cấu hình baseline chuẩn bị quét lr:")
print({k: BASE[k] for k in ("optimizer", "lr", "batch", "epochs", "hidden", "init", "precision")})

In [ ]:
# ---- Quét lr cho baseline, chọn bằng VAL ----
LR_SCAN = [0.01, 0.03, 0.1]        # GUIDE: SGD+momentum quanh 0.01–0.1
for lr in LR_SCAN:
    run_exp({**BASE, "exp_id": f"opt-sgdm-lr{lr:g}", "group": "optimizer",
             "description": f"SGD+momentum 0.9, lr = {lr:g}",
             "lr": lr, "notes": "Quét lr trên val; arm SGD+mom của phép so sánh bộ tối ưu."})
scan = pd.DataFrame([dict(lr=lr, val_macro_f1=round(RESULTS[f"opt-sgdm-lr{lr:g}"]["summary"]["val_macro_f1"], 4),
                          val_acc=round(RESULTS[f"opt-sgdm-lr{lr:g}"]["summary"]["val_acc"], 4),
                          best_epoch=RESULTS[f"opt-sgdm-lr{lr:g}"]["summary"]["best_epoch"],
                          s_per_epoch=round(RESULTS[f"opt-sgdm-lr{lr:g}"]["summary"]["time_per_epoch_s"], 2))
                     for lr in LR_SCAN])
display(scan)
BASE_LR = float(scan.loc[scan["val_macro_f1"].idxmax(), "lr"])
print(f"\n==> lr được chọn cho baseline (theo val macro-F1): {BASE_LR:g}")
print("    (chọn bằng val; tập eval chưa vào bước này)")

In [ ]:
# ---- Baseline với 3 seed để đo độ nhiễu ----
BASE_SEEDS = [1, 2, 3]
for s in BASE_SEEDS:
    run_exp({**BASE, "exp_id": f"base-s{s}", "lr": BASE_LR, "seed": s,
             "description": f"Baseline M-base, lr = {BASE_LR:g}, seed {s}",
             "notes": "Cùng cấu hình, chỉ khác seed -> đo độ nhiễu."})

rows_all = [to_row(RESULTS[e]) for e in RESULTS]
NOISE = noise_stats(rows_all, [f"base-s{s}" for s in BASE_SEEDS])
print("ĐỘ NHIỄU GIỮA CÁC SEED của baseline:")
for m in ("val_macro_f1", "val_acc", "best_val_loss"):
    d = NOISE[m]
    print(f"  {m:14s} các giá trị {[round(v,5) for v in d['values']]}  "
          f"TB = {d['mean']:.5f}  σ = {d['std']:.5f}  2σ = {d['two_sigma']:.5f}")
print(f"\n==> NGƯỠNG NHIỄU 2σ (val macro-F1) = {NOISE['val_macro_f1']['two_sigma']:.5f}")
print(f"    accuracy 'luôn đoán lớp đa số' trên val = {data['majority_acc_val']:.4f}  <- mô hình phải vượt")
display(brief([f"base-s{s}" for s in BASE_SEEDS]))
show_fig("base-s1")
plot_compare([RESULTS[f"base-s{s}"] for s in BASE_SEEDS], "val_loss",
             f"{FIGS_DIR}/compare_seed.png", "Baseline: độ nhiễu giữa 3 seed")
show_fig("compare_seed")

---
## Part 3 — Các thí nghiệm

Mỗi thí nghiệm: **dự đoán trước** (viết ở ô markdown ngay trên) → đổi **một** yếu tố so với
baseline → chạy → **đối chiếu** (ô markdown ngay dưới). Chủ đề trong 7 nhóm:
`loss` · `optimizer` · `hparam` · `dropout` · `clipping` · `amp` · `init`.

Ngưỡng nhiễu dùng để kết luận: **2σ = x.xxxxx** trên `val macro-F1`
(tính ở Part 2 từ 3 seed baseline). Chênh lệch nhỏ hơn 2σ **không phải bằng chứng**.

### 3.1 Hàm mất mát — CE vs MSE

**Dự đoán trước khi chạy.** Ở cùng `lr` và cùng kiến trúc, MSE trên nhãn one-hot sẽ học **chậm hơn**
và cho macro-F1 thấp hơn cross-entropy. Lý do: gradient của CE theo logit là `p − onehot`, tức
`softmax(CE) − y`, sai ở đâu phạt nặng đúng chỗ đó; gradient của MSE là `2/N · (z − y)·σ(z)`,
bị nhân với `σ'(z)` nên **bão hoà** (khi `σ(z)` bão hoà thì gradient gần như triệt tiêu), và
vì không có lũy thừa nên tín hiệu cho lớp hiếm rất yếu. Ngoài ra loss của MSE ở thang khác
(nhỏ hơn nhiều) nên **không được so loss trực tiếp**, phải so accuracy và macro-F1.

In [ ]:
# ---- 3.1 CE (baseline) vs MSE, chỉ đổi đúng yếu tố `loss` ----
run_exp({**BASE, "lr": BASE_LR, "exp_id": "loss-mse", "group": "loss",
         "description": "MSE trên nhãn one-hot (thay CE)", "loss": "mse",
         "notes": "Dự đoán: macro-F1 thấp hơn CE. MSE dùng nn.MSELoss (không hệ số 1/2, "
                  "trung bình trên mọi phần tử B x 7) nên thang đo khác CE — so bằng metric."})
display(brief(["base-s1", "loss-mse"]))
show_fig("loss-mse")
plot_compare([RESULTS["base-s1"], RESULTS["loss-mse"]], "val_macro_f1",
             f"{FIGS_DIR}/compare_loss.png", "3.1 CE vs MSE (val macro-F1)")
show_fig("compare_loss")

**Đối chiếu 3.1.** Xem `compare_loss.png` và cột `delta_vs_base` trong bảng trên: MSE cho
macro-F1 thấp hơn baseline ở cùng `lr` — khớp hay khác dự đoán? Nhớ nhắc: hai loss khác thang đo
(giá trị loss không so được), còn gradient thì MSE bị nhân bởi đạo hàm sigmoid (bão hoà) và
tín hiệu lớp hiếm bị loãng khi chia trung bình trên 7 thành phần.

### 3.2 Bộ tối ưu hoá

**Dự đoán trước khi chạy.** Mỗi bộ tối ưu phải được chỉnh `lr` riêng thì mới so sánh công bằng.
Dự đoán: **Adam/AdamW thắng SGD+SGD+momentum rõ rệt** ở số epoch cố định (20), vì chúng chia bước
theo `m̂/(√v̂+ε)` nên mỗi tham số có bước đi gần như cùng một thang, hội tụ nhanh và ít nhạy `lr` hơn;
ngược lại **SGD nhạy `lr` nhất** và dễ dao động. Ở 20 epoch (chưa hội tụ hẳn) lợi thế hội tụ nhanh
của Adam thường thắng, nhưng khoảng cách sẽ **thu hẹp** khi huấn luyện lâu hơn.
Với `weight_decay = 0`, Adam và AdamW phải cho **cùng** kết quả (cùng công thức, chỉ khác tên chỗ suy giảm trọng số).

In [ ]:
# ---- 3.2 Mỗi bộ tối ưu được thử nhiều lr, rồi so ở lr tốt nhất của chính nó ----
# lr lấy từ GUIDE (khối tham chiếu, chưa kiểm chứng trước): SGD ~0.01–0.1; Adam/AdamW ~1e-4–3e-3
OPT_ARMS = [
    dict(key="sgd",          optimizer="sgd",          lrs=[0.01, 0.03, 0.1],  wd=0.0),
    dict(key="sgd_momentum", optimizer="sgd_momentum", lrs=[0.01, 0.03, 0.1],  wd=0.0),
    dict(key="adam",         optimizer="adam",         lrs=[3e-4, 1e-3, 3e-3], wd=0.0),
    dict(key="adamw",        optimizer="adamw",        lrs=[3e-4, 1e-3, 3e-3], wd=0.01),
]
OPT_PREFIX = {"sgd": "sgd", "sgd_momentum": "sgdm", "adam": "adam", "adamw": "adamw"}

for arm in OPT_ARMS:
    for lr in arm["lrs"]:
        eid = f"opt-{OPT_PREFIX[arm['key']]}-lr{lr:g}"
        if eid in RESULTS:                      # arm SGD+mom đã chạy ở phần quét lr
            continue
        run_exp({**BASE, "lr": BASE_LR, "exp_id": eid, "group": "optimizer",
                 "optimizer": arm["optimizer"], "weight_decay": arm["wd"], "lr": lr,
                 "description": f"{arm['optimizer']} (wd={arm['wd']:g}), lr = {lr:g}",
                 "notes": "So sánh bộ tối ưu: mỗi bộ thử 3 lr cách nhau ~3–10 lần."})

# Kiểm chứng: Adam vs AdamW khi weight_decay = 0 phải cho cùng kết quả
run_exp({**BASE, "lr": BASE_LR, "exp_id": "opt-adamw-wd0-lr0.001", "group": "optimizer",
         "optimizer": "adamw", "weight_decay": 0.0, "lr": 1e-3,
         "description": "AdamW với weight_decay = 0 (kiểm chứng tương đương Adam)",
         "notes": "Với wd=0, Adam và AdamW có cùng công thức cập nhật -> kết quả phải khớp."})

all_opt = [e for e in RESULTS if e.startswith("opt-")]
display(brief(all_opt))

print("\nSo ở lr TỐT NHẤT của từng bộ tối ưu:")
best_per_arm = {}
for arm in OPT_ARMS:
    ids = [f"opt-{OPT_PREFIX[arm['key']]}-lr{lr:g}" for lr in arm["lrs"]]
    ids = [e for e in ids if e in RESULTS]
    b = max(ids, key=lambda e: RESULTS[e]["summary"]["val_macro_f1"])
    best_per_arm[arm["key"]] = b
    print(f"  {arm['optimizer']:<14s} lr tốt nhất = {RESULTS[b]['cfg']['lr']:<8g} "
          f"{b:<24s} val macro-F1 = {RESULTS[b]['summary']['val_macro_f1']:.4f}  "
          f"(best epoch {RESULTS[b]['summary']['best_epoch']})")
BEST_OPT_KEY = max(best_per_arm, key=lambda k: RESULTS[best_per_arm[k]]["summary"]["val_macro_f1"])
BEST_OPT_EID = best_per_arm[BEST_OPT_KEY]
BEST_OPT_CFG = RESULTS[BEST_OPT_EID]["cfg"]
print(f"\n==> Bộ tối ưu thắng (chọn bằng val): {BEST_OPT_CFG['optimizer']} lr = {BEST_OPT_CFG['lr']:g} "
      f"({BEST_OPT_EID})")

for arm in OPT_ARMS:
    ids = [f"opt-{OPT_PREFIX[arm['key']]}-lr{lr:g}" for lr in arm["lrs"] if f"opt-{OPT_PREFIX[arm['key']]}-lr{lr:g}" in RESULTS]
    if len(ids) > 1:
        plot_compare([RESULTS[e] for e in ids], "val_macro_f1", f"{FIGS_DIR}/compare_opt_{arm['key']}.png",
                     f"3.2 {arm['optimizer']}: nhạy cảm với lr (val macro-F1)")
        show_fig(f"compare_opt_{arm['key']}")
plot_compare([RESULTS[e] for e in all_opt], "val_loss", f"{FIGS_DIR}/compare_optimizer.png",
             "3.2 Toàn bộ bộ tối ưu: val loss")
show_fig("compare_optimizer")

**Đối chiếu 3.2.** Điền vào báo cáo: mỗi bộ ở `lr` tốt nhất (bảng in ở trên), độ nhạy với `lr`
(biên độ giữa `lr` tệ nhất và tốt nhất), và cơ chế: SGD có phương sai gradient lớn nên cần `lr` nhỏ
và momentum; Adam chuẩn hoá theo phương sai giới hạn tích cực nên ổn định hơn nhưng bước cập nhật
"không có quy mô" nên hội tụ chậm ở cuối. Ở 20 epoch, epoch tốt nhất có còn là epoch cuối không?

### 3.3 Hyper-parameter

**Dự đoán trước khi chạy.**

- **Batch nhỏ (128)**: nhiều bước cập nhật hơn gấp 4 ở cùng số epoch → học nhanh hơn, nhưng nhiễu
  gradient lớn hơn và tốn thời gian/epoch gấp 4. **Batch lớn (2048)**: ít bước hơn 4 lần → ở cùng
  20 epoch chắc chắn *chưa* hội tụ (best epoch sẽ là epoch cuối), và thường cần tăng `lr` để bù
  (quy tắc lô ×k thì η ×k) — ta giữ nguyên `lr` để cô lập đúng một yếu tố.
- **Độ rộng / độ sâu**: `M-wide` (161k tham số) nên vẫn dưới-khớp ở 20 epoch → dự đoán tốt hơn
  `M-base`; `M-deep` thêm ít tham số nên gần như không khác.
- **Cosine annealing**: giảm `lr` về 0 giúp "hạ cánh" nên thường cải thiện một chút ở ngân sách epoch cố định.
- **Huấn luyện lâu hơn (60 epoch)**: vì ở 20 epoch đường val loss vẫn còn giảm (best epoch = epoch cuối),
  chạy lâu hơn phải cho macro-F1 cao hơn rõ rệt — đây là bằng chứng mô hình **chưa** hội tụ chứ không phải quá khớp.

In [ ]:
# ---- 3.3 Hyper-parameter: mỗi lần chỉ đổi MỘT yếu tố ----
HPARAMS = [
    dict(exp_id="hparam-batch128",  description="Batch size 128 (thay 512)",        batch=128),
    dict(exp_id="hparam-batch2048", description="Batch size 2048 (thay 512)",       batch=2048),
    dict(exp_id="hparam-wd1e-4",    description="weight_decay 1e-4 (thay 0)",        weight_decay=1e-4),
    dict(exp_id="hparam-cosine",    description="Cosine annealing theo bước",       scheduler="cosine"),
    dict(exp_id="arch-wide",        description="M-wide 54->512->256->7",            hidden=(512, 256)),
    dict(exp_id="arch-deep",        description="M-deep 54->256->128->64->7",        hidden=(256, 128, 64)),
    dict(exp_id="hparam-epoch%d" % EPOCHS_LONG,
         description=f"{EPOCHS_LONG} epoch (thay {EPOCHS}) — kiểm tra mô hình đã hội tụ chưa",
         epochs=EPOCHS_LONG),
]
for h_ in HPARAMS:
    run_exp({**BASE, "lr": BASE_LR, "group": "hparam", **h_,
             "notes": "Chỉ đổi một yếu tố so với baseline (lr giữ nguyên). "
                      "Số epoch khác nghĩa là số bước cập nhật khác — cần nhớ khi giải thích."})

display(brief(["base-s1"] + [h_["exp_id"] for h_ in HPARAMS if h_["exp_id"] in RESULTS]))
plot_compare([RESULTS["base-s1"]] + [RESULTS[h_["exp_id"]] for h_ in HPARAMS if h_["exp_id"] in RESULTS],
             "val_macro_f1", f"{FIGS_DIR}/compare_hparam.png", "3.3 Hyper-parameter (val macro-F1)")
show_fig("compare_hparam")

**Đối chiếu 3.3.** Nhớ phân biệt **số bước cập nhật**: `371847/128 ≈ 2904` bước/epoch so với
`182` bước/epoch ở batch 2048 — cùng 20 epoch nhưng khác số bước rất xa, nên so sánh phải tính cả
thời gian/epoch (`s_per_epoch`) và số bước. `M-wide`/`M-deep` đổi **kiến trúc** (số tham số khác
hẳn) — đó là "một yếu tố" ở đây. Với `hparam-epoch*`: best epoch có còn là epoch cuối không, và
khoảng cách `val_loss − train_loss` có tăng (dấu hiệu bắt đầu quá khớp) không?

### 3.4 Dropout

**Dự đoán trước khi chạy.** Ở 20 epoch trên tập 371k mẫu với mạng chỉ 48k tham số, mô hình
**chưa quá khớp** (khoảng cách `val_loss − train_loss` còn nhỏ), nên dropout `q = 0.3` và `q = 0.5`
nhiều khả năng **không giúp** và còn làm chậm hội tụ; chỉ `q` rất nhỏ (`0.1`) là ứng viên hợp lý.
Đây là bài học: dropout là **thuốc cho quá khớp**, dùng khi đã có bằng chứng quá khớp.

In [ ]:
# ---- 3.4 Dropout: q = 0.1 / 0.2 / 0.5 (baseline q = 0) ----
run_exp({**BASE, "lr": BASE_LR, "exp_id": "drop-0.1", "group": "dropout", "dropout": 0.1,
         "description": "Dropout q = 0.1", "notes": "Dự đoán: q nhỏ có thể giúp nhẹ."})
run_exp({**BASE, "lr": BASE_LR, "exp_id": "drop-0.2", "group": "dropout", "dropout": 0.2,
         "description": "Dropout q = 0.2",
         "notes": "Dự đoán: không giúp vì mô hình chưa quá khớp."})
run_exp({**BASE, "lr": BASE_LR, "exp_id": "drop-0.5", "group": "dropout", "dropout": 0.5,
         "description": "Dropout q = 0.5",
         "notes": "Dự đoán: quá mạnh, macro-F1 giảm rõ."})

display(brief(["base-s1", "drop-0.1", "drop-0.2", "drop-0.5"]))
plot_compare([RESULTS[e] for e in ("base-s1", "drop-0.1", "drop-0.2", "drop-0.5")],
             "val_loss", f"{FIGS_DIR}/compare_dropout.png",
             "3.4 Dropout: khoảng cách train loss - val loss")
show_fig("compare_dropout")

**Đối chiếu 3.4.** Vẽ chồng cho thấy khoảng cách `train_loss` (đo ở `eval()`) với `val_loss`
**hẹp lại hay rộng ra** khi tăng `q`. Lưu ý: nếu bạn đo train loss **trong lúc huấn luyện** (dropout
đang bật) thì sẽ thấy dropout "làm tăng train loss" một cách giả — đó chỉ là hiệu ứng tắt nơ-ron.

### 3.5 Cắt gradient

`g ← g · min(1, c/‖g‖)` với `‖g‖` là chuẩn L2 **toàn cục** của mọi gradient gộp lại.

**Dự đoán trước khi chạy.** Ở `lr` của baseline, `‖g‖` ổn định và nhỏ hơn 1, nên chọn `c = 1.0`
sẽ **không** (hoặc rất hiếm khi) kích hoạt → chênh lệch so với baseline chỉ là nhiễu, và đó chính là
kết quả đúng đắn phải báo cáo. Vì thế thí nghiệm **phản chứng** mới là thứ chứng minh clipping có ích:
tăng `lr` lên `10 × lr` cho tới khi gradient nổ gai và loss dao động/NaN, rồi chạy lại **cùng lr đó
có clip**. Dự đoán: không clip thì loss phá vỡ (NaN hoặc macro-F1 về mức đoán đa số), có clip thì vẫn học được.

In [ ]:
# ---- 3.5 Chọn c dựa trên grad_norm quan sát được ở baseline ----
h0 = RESULTS["base-s1"]["history"]
gn_mean = np.array(h0["grad_norm"]); gn_max = np.array(h0["grad_norm_max"]); gn_min = np.array(h0["grad_norm_min"])
print("grad_norm của baseline (đo TRƯỚC khi clip):")
print(f"  theo epoch — min {gn_min.min():.3f} / trung vị {gn_mean.mean():.3f} / max {gn_max.max():.3f}")
print(f"  trung vị của trung bình epoch: {np.median(gn_mean):.3f};  p90 của max trong epoch: {np.percentile(gn_max, 90):.3f}")

CLIP_C = 1.0
frac = float(((gn_min < CLIP_C) & (gn_max > CLIP_C)).mean())
print(f"\n==> chọn c = {CLIP_C:g}: trong {100*frac:.0f}% epoch có ít nhất một phần bước bị cắt "
      f"(epoch có min < c < max).")
print("    Nếu c lớn hơn mọi grad_norm thì clipping không làm gì cả — kết luận 'không khác' chỉ là nhiễu.")

run_exp({**BASE, "lr": BASE_LR, "exp_id": "clip-1.0", "group": "clipping", "clip_norm": CLIP_C,
         "description": f"Gradient clipping c = {CLIP_C:g} ở lr baseline",
         "notes": f"Dự đoán: clipping hiếm khi kích hoạt ở lr thường nên chênh lệch chỉ là nhiễu. c={CLIP_C:g} chọn từ grad_norm của baseline."})

# --- thí nghiệm phản chứng: lr cao, có clip và không clip
HIGH_LR_MULT = 10
HIGH_LR = BASE_LR * HIGH_LR_MULT
run_exp({**BASE, "lr": HIGH_LR, "exp_id": "clip-hilr-noclip", "group": "clipping",
         "description": f"lr cao {HIGH_LR:g} (= {HIGH_LR_MULT}×lr baseline), KHÔNG clip",
         "notes": "Phản chứng: gradient nổ gai ở lr cao, kỳ vọng loss dao động/NaN."})
run_exp({**BASE, "lr": HIGH_LR, "exp_id": "clip-hilr-clip1.0", "group": "clipping", "clip_norm": CLIP_C,
         "description": f"lr cao {HIGH_LR:g} (= {HIGH_LR_MULT}×lr baseline), CÓ clip c = {CLIP_C:g}",
         "notes": "Cùng lr cao nhưng có clip — so với 'clip-hilr-noclip' để thấy vai trò của clipping."})

display(brief(["base-s1", "clip-1.0", "clip-hilr-noclip", "clip-hilr-clip1.0"]))
plot_compare([RESULTS[e] for e in ("base-s1", "clip-1.0", "clip-hilr-noclip", "clip-hilr-clip1.0")],
             "grad_norm", f"{FIGS_DIR}/compare_clipping.png",
             f"3.5 grad_norm (trước clip) — ngưỡng c = {CLIP_C:g}")
show_fig("compare_clipping")

**Đối chiếu 3.5.** Ở `lr` thường, clipping có kích hoạt không (dựa vào `grad_norm`) và chênh lệch có
vượt 2σ không — nếu không vượt, kết luận phải là "clipping không có tác dụng ở `lr` này", **không phải**
"clipping vô dụng". Ở `lr` cao: cột `diverged` và đường `grad_norm` (thang log) cho thấy cắt gradient
cứu được huấn luyện ra sao — cơ chế là bước cập nhật không bao giờ vượt quá `c·lr` theo mỗi hướng.

### 3.6 Mixed precision

FP16 dùng `torch.autocast(dtype=torch.float16)` + `GradScaler`; BF16 dùng `dtype=torch.bfloat16`
(thường **không** cần `GradScaler`). Tham số vẫn FP32 — autocast chỉ hạ độ chính xác phép toán.

**Dự đoán trước khi chạy.** Với mạng **rất nhỏ** (48k tham số, batch 512), thời gian mỗi bước bị
chi phối bởi **chi phí gọi kernel và chuyển đổi dtype**, không phải FLOPs → mixed precision có thể
**không nhanh hơn** (thậm chí chậm hơn) FP32. Bộ nhớ GPU giảm nhẹ. Chất lượng phải gần như không đổi.
FP16 cần `GradScaler` vì dải số động của FP16 hẹp (~5·10⁻⁵ … 65504): gradient rất nhỏ sẽ tràn về 0,
nên loss phải được nhân với hệ số scale rồi bỏ scale trước khi cắt gradient. BF16 có cùng độ rộng
thập phân như FP32 (8 bit mantissa) nên không cần scale.

In [ ]:
# ---- 3.6 FP32 vs FP16 vs BF16 ----
bf16_hw = bool(device == "cuda" and torch.cuda.is_bf16_supported())
run_exp({**BASE, "lr": BASE_LR, "exp_id": "amp-fp16", "group": "amp", "precision": "fp16",
         "clip_norm": CLIP_C,
         "description": "FP16 autocast + GradScaler (fp32: 0 clip)",
         "notes": "Dự đoán: không nhanh hơn FP32 vì mạng nhỏ; chất lượng gần như không đổi."})
run_exp({**BASE, "lr": BASE_LR, "exp_id": "amp-bf16", "group": "amp", "precision": "bf16",
         "clip_norm": CLIP_C,
         "description": "BF16 autocast (không GradScaler)",
         "notes": ("GPU có phần cứng BF16." if bf16_hw else
                   "GPU KHÔNG có phần cứng BF16 (ví dụ T4/P100) -> chạy mô phỏng phần mềm, "
                   "thường chậm hơn cả FP32; vẫn ghi kết quả để minh bạch.")})

amp_df = brief(["base-s1", "amp-fp16", "amp-bf16"])
display(amp_df)
print("\nThời gian / bộ nhớ / độ chính xác so với FP32:")
base_t = RESULTS["base-s1"]["summary"]
for e in ("amp-fp16", "amp-bf16"):
    s = RESULTS[e]["summary"]
    print(f"  {e:10s} {s['time_per_epoch_s']:6.2f} s/epoch ({s['time_per_epoch_s']/base_t['time_per_epoch_s']:5.2f}× fp32)"
          f" | peak {s['peak_mem_MB']:6.1f} MB ({s['peak_mem_MB']/max(base_t['peak_mem_MB'],1e-9):5.2f}×)"
          f" | val macro-F1 {s['val_macro_f1']:.4f} (Δ = {s['val_macro_f1']-base_t['val_macro_f1']:+.4f})")
plot_compare([RESULTS[e] for e in ("base-s1", "amp-fp16", "amp-bf16")], "val_loss",
             f"{FIGS_DIR}/compare_amp.png", "3.6 Mixed precision (val loss)")
show_fig("compare_amp")

### 3.7 Khởi tạo tham số

| init | công thức | PyTorch |
|---|---|---|
| `zeros` | `W = 0` | `nn.init.zeros_` |
| `normal` | `W ~ N(0, 0.01²)` | `nn.init.normal_` |
| `xavier` | `Var[W] = 2/(n_in+n_out)` | `nn.init.xavier_normal_` |
| `he` | `Var[W] = 2/n_in` | `nn.init.kaiming_normal_(w, nonlinearity="relu")` |
| `default` | mặc định `nn.Linear` (uniform ±1/√n_in, **không phải He**) | không làm gì |

**Dự đoán trước khi chạy.**

- `zeros`: mọi nơ-ron cùng lớp nhận **cùng một** gradient → W giữ nguyên tính đối xứng → mạng
  không học được (tương đương một nơ-ron duy nhất). Thêm nữa ReLU(0) = 0 nên gradient qua tầng ẩn
  bị chặn hoàn toàn. Dự đoán: val macro-F1 ≈ 0.09 (mức đoán đa số).
- `normal (std=0.01)`: tín hiệu bị thu nhỏ qua mỗi tầng → độ lệch chuẩn kích hoạt giảm dần theo tầng,
  học chậm nhưng vẫn học được.
- `xavier` giữ phương sai qua tầng nhưng tính cả nhiều chiều (`2/(n_in+n_out)`) nên **giảm** phương sai
  kích hoạt; `he` chỉ dùng `n_in` (`2/n_in`) nên **bù đúng** phần tắt một nửa của ReLU → phương sai
  kích hoạt giữ gần như không đổi qua các tầng. Với ReLU, He đúng hơn Xavier.
- `default` có phương sai `1/(3·n_in)` — nhỏ hơn He một nửa → tín hiệu yếu dần.

In [ ]:
# ---- 3.7 Khởi tạo: zeros / normal / xavier / default (baseline là He) ----
INITS = [
    ("init-zeros",  "zeros",  "W = 0 — mất tính đối xứng, ReLU(0) chặn gradient"),
    ("init-normal", "normal", "W ~ N(0, 0.01^2) — tín hiệu nhỏ, học chậm"),
    ("init-xavier", "xavier", "xavier_normal_: Var = 2/(n_in+n_out)"),
    ("init-default", "default", "mặc định nn.Linear: uniform, Var = 1/(3 n_in) — KHÔNG phải He"),
]
stat_rows = []
for eid, init, desc in INITS:
    run_exp({**BASE, "lr": BASE_LR, "exp_id": eid, "group": "init", "init": init,
             "description": f"Khởi tạo {init}", "notes": desc})
    stds = activation_stats(MLP((256, 128), init=init).to(device), data["X_val"][:20_000])
    s = RESULTS[eid]["summary"]
    stat_rows.append(dict(exp_id=eid, init=init,
                          step0_loss=round(s["step0_loss"], 4),
                          **{f"std_tầng{i+1}": round(v, 4) for i, v in enumerate(stds[:2])},
                          std_logits=round(stds[2], 4),
                          val_macro_f1=round(s["val_macro_f1"], 4),
                          val_acc=round(s["val_acc"], 4),
                          s_per_epoch=round(s["time_per_epoch_s"], 2)))
print("Độ lệch chuẩn kích hoạt ở bước 0 (một lô val) và kết quả sau 20 epoch:")
display(pd.DataFrame(stat_rows))
print("Baseline (he): " + str({f"std_tầng{i+1}": v for i, v in
      enumerate(activation_stats(MLP((256, 128), init="he").to(device), data["X_val"][:20_000])[:2])}))
display(brief(["base-s1"] + [e for e, _, _ in INITS]))
plot_compare([RESULTS["base-s1"]] + [RESULTS[e] for e, _, _ in INITS],
             "val_macro_f1", f"{FIGS_DIR}/compare_init.png", "3.7 Khởi tạo tham số (val macro-F1)")
show_fig("compare_init")

**Đối chiếu 3.7.** Với `zeros`, gradient của các nơ-ron trong cùng một tầng có **bằng nhau không**?
(hãy in ra để trả lời bằng số liệu). Mạng 3 tầng có đủ sâu để thấy rõ khác biệt phương sai kích
hoạt giữa `he` và `xavier` không, hay phải dùng mạng sâu hơn nhiều mới thấy?

---
## 3.8 Cấu hình cuối cùng (chọn **bằng val**)

Ghép các lựa chọn đã được val ủng hộ: bộ tối ưu + `lr` tốt nhất (3.2), `dropout` (3.4), và số epoch
lớn hơn (3.3). **Không** dùng tập eval ở mọi bước của phần này.

In [ ]:
# ---- Cấu hình cuối cùng: chỉ ghép những gì đã có bằng chứng từ val ----
Q_DROP = 0.1 if RESULTS["drop-0.1"]["summary"]["val_macro_f1"] >= RESULTS["base-s1"]["summary"]["val_macro_f1"] else 0.0
FINAL_BASE = {**BASE,
              "optimizer": BEST_OPT_CFG["optimizer"],
              "lr": BEST_OPT_CFG["lr"],
              "weight_decay": BEST_OPT_CFG["weight_decay"],
              "momentum": BEST_OPT_CFG["momentum"],
              "dropout": Q_DROP,
              "epochs": EPOCHS_FINAL}
print("Cấu hình cuối cùng (ghép từ val):")
print({k: FINAL_BASE[k] for k in ("optimizer", "lr", "weight_decay", "dropout", "batch", "epochs", "hidden", "init")})
print("Các yếu tố này đổi so với baseline: optimizer+lr (3.2), dropout (3.4), số epoch (3.3).")

FINAL_SEEDS = [1, 2, 3]
for s in FINAL_SEEDS:
    run_exp({**FINAL_BASE, "exp_id": f"final-s{s}", "group": "final", "seed": s,
             "description": f"Cấu hình cuối cùng, seed {s}",
             "notes": "Ghép các lựa chọn có bằng chứng từ val; chạy 3 seed để báo cáo trung bình ± σ."})
display(brief(["base-s1"] + [f"final-s{s}" for s in FINAL_SEEDS]))

f_stat = noise_stats([to_row(RESULTS[e]) for e in [f"final-s{s}" for s in FINAL_SEEDS]],
                     [f"final-s{s}" for s in FINAL_SEEDS])
print("\nCấu hình cuối cùng qua 3 seed:")
print(f"  val macro-F1 = {f_stat['val_macro_f1']['mean']:.4f} ± {f_stat['val_macro_f1']['std']:.4f} (σ)")
print(f"  val accuracy = {f_stat['val_acc']['mean']:.4f} ± {f_stat['val_acc']['std']:.4f} (σ)")
b_stat = NOISE
print(f"  baseline      = {b_stat['val_macro_f1']['mean']:.4f} ± {b_stat['val_macro_f1']['std']:.4f} (σ)")
delta = f_stat['val_macro_f1']['mean'] - b_stat['val_macro_f1']['mean']
print(f"  Δ = {delta:+.4f}  |  2σ_baseline = {b_stat['val_macro_f1']['two_sigma']:.4f}  ->  "
      f"{'VƯỢT nhiễu' if abs(delta) > b_stat['val_macro_f1']['two_sigma'] else 'CHƯA vượt nhiễu'}")
show_fig("final-s1")

---
## Part 4 — Đánh giá cuối trên tập eval, bảng và phân tích lỗi

Cấu hình đã chọn **bằng val** ở trên. Bây giờ mới dùng tập `eval` (116 203 mẫu), và **chỉ dùng
đúng một lần cho mỗi cấu hình**: baseline và cấu hình cuối cùng.

In [ ]:
# ---- Dự đoán trên toàn bộ eval, cho baseline và cấu hình cuối cùng ----
# (chạy lại từ đầu vì best_state không lưu trong results/*.json)
res_base   = run_experiment({**BASE,   "lr": BASE_LR, "seed": 1}, data)
res_final  = run_experiment({**FINAL_BASE, "exp_id": "final-s1", "seed": 1}, data)
RESULTS["base-s1"], RESULTS["final-s1"] = res_base, res_final
save_result(res_base, RES_DIR);  save_result(res_final, RES_DIR)
plot_run(res_base,  f"{FIGS_DIR}/base-s1.png")
plot_run(res_final, f"{FIGS_DIR}/final-s1.png")

_ = final_eval({**BASE, "lr": BASE_LR, "exp_id": "base-s1"}, res_base,  data,
                f"{OUT_DIR}/predictions_eval_baseline.csv")
_ = final_eval({**FINAL_BASE, "exp_id": "final-s1"},               res_final, data,
                f"{OUT_DIR}/predictions_eval.csv")

In [ ]:
# ---- Chấm bằng đúng scripts/evaluate.py ----
for name, out in (("predictions_eval_baseline.csv", "eval_result_baseline.json"),
                  ("predictions_eval.csv", "eval_result.json")):
    subprocess.run([sys.executable, os.path.join(REPO_ROOT, "scripts", "evaluate.py"),
                    "--pred", os.path.join(OUT_DIR, name),
                    "--data", os.path.join(REPO_ROOT, "data", "covtype.csv.gz"),
                    "--meta", os.path.join(REPO_ROOT, "data", "split_metadata.csv"),
                    "--out", os.path.join(OUT_DIR, out)], cwd=REPO_ROOT, check=True, env=SUBENV)

EVAL_BASE  = json.load(open(f"{OUT_DIR}/eval_result_baseline.json", encoding="utf-8"))
EVAL_FINAL = json.load(open(f"{OUT_DIR}/eval_result.json", encoding="utf-8"))
print("\n" + "=" * 78)
print(f"{'cấu hình':<26s} {'val macro-F1':>13s} {'EVAL macro-F1':>14s} {'EVAL accuracy':>14s}")
print("-" * 78)
print(f"{'baseline (base-s1)':<26s} {res_base['summary']['val_macro_f1']:>13.4f} "
      f"{EVAL_BASE['macro_f1']:>14.4f} {EVAL_BASE['accuracy']:>14.4f}")
print(f"{'cấu hình cuối cùng':<26s} {res_final['summary']['val_macro_f1']:>13.4f} "
      f"{EVAL_FINAL['macro_f1']:>14.4f} {EVAL_FINAL['accuracy']:>14.4f}")
print(f"{'cải thiện (eval)':<26s} {'':>13s} "
      f"{EVAL_FINAL['macro_f1'] - EVAL_BASE['macro_f1']:>+14.4f}")
print(f"{'chênh lệch val - eval (final)':<26s} "
      f"{res_final['summary']['val_macro_f1'] - EVAL_FINAL['macro_f1']:>+13.4f}")
print("=" * 78)
print("nếu val và eval lệch nhau nhiều, nguyên nhân có thể là: chọn best epoch trên val bị overfit "
      "slight lên val, hoặc val nhỏ hơn/thống kê khác eval.")

In [ ]:
# ---- 4b: phân tích lỗi theo lớp trên eval (từ eval_result.json) ----
pc = pd.DataFrame(EVAL_FINAL["per_class"])
pc.columns = ["cls", "support", "precision", "recall", "f1"]
display(pc.round(4))
cm = np.array(EVAL_FINAL["confusion_matrix"])
NAME = {0: "Spruce/Fir", 1: "Lodgepole Pine", 2: "Ponderosa Pine", 3: "Cottonwood/Willow",
        4: "Aspen", 5: "Douglas-fir", 6: "Krummholz"}
print("ma trận nhầm lẫn trên eval (hàng = nhãn thật, cột = dự đoán):")
display(pd.DataFrame(cm, index=[f"{i} {NAME[i]}" for i in range(7)],
                     columns=[str(i) for i in range(7)]))

fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.8))
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)     # chuẩn hoá theo hàng = recall
im = ax[0].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax[0].set_xticks(range(7)); ax[0].set_yticks(range(7))
ax[0].set_xticklabels(range(7)); ax[0].set_yticklabels(range(7))
ax[0].set_xlabel("dự đoán"); ax[0].set_ylabel("nhãn thật"); ax[0].set_title("Ma trận nhầm lẫn (chuẩn hoá theo hàng)")
for i in range(7):
    for j in range(7):
        ax[0].text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=7,
                   color="white" if cmn[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax[0], fraction=0.046)
x = np.arange(7); w = 0.27
ax[1].bar(x - w, pc["precision"], w, label="precision")
ax[1].bar(x,     pc["recall"],    w, label="recall")
ax[1].bar(x + w, pc["f1"],        w, label="F1")
ax[1].set_xticks(x); ax[1].set_xticklabels([f"{i}\n{NAME[i][:9]}" for i in range(7)], fontsize=7)
ax[1].set_ylim(0, 1.05); ax[1].set_ylabel("score"); ax[1].set_title("Precision / Recall / F1 theo lớp (eval)")
ax[1].legend(fontsize=8); ax[1].grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(f"{FIGS_DIR}/error_analysis_eval.png", dpi=130, bbox_inches="tight")
plt.show()

worst = int(pc.loc[pc["f1"].idxmin(), "cls"])
conf = [(j, int(cm[worst, j])) for j in range(7) if j != worst]
conf.sort(key=lambda t: -t[1])
print(f"\nLớp khó nhất: lớp {worst} ({NAME[worst]}) — F1 = {pc.loc[pc['cls']==worst,'f1'].iat[0]:.4f}, "
      f"support = {int(pc.loc[pc['cls']==worst,'support'].iat[0])}")
print("bị nhầm nhiều nhất với:", ", ".join(f"lớp {j} ({NAME[j]}): {n} mẫu" for j, n in conf[:3]))
print("recall thấp nhất:", int(pc.loc[pc['recall'].idxmin(),'cls']),
      "| precision thấp nhất:", int(pc.loc[pc['precision'].idxmin(),'cls']))

**Phân tích lỗi (viết bằng lời của bạn):** lớp khó nhất là lớp ... vì ...
(số mẫu ít nhất thì recall dễ thấp; hai lớp có đặc trưng gần nhau thì dễ nhầm hai chiều —
nhìn số ở hàng `worst` và cột `worst` của ma trận). Một cách cải thiện sẽ thử: ...

In [ ]:
# ---- 4a: điền experiments.xlsx ----
NOTES = {e: RESULTS[e]["cfg"].get("notes", "") for e in RESULTS}

# Sắp xếp theo thứ tự nhóm để bảng dễ đọc
GROUP_ORDER = ["baseline", "loss", "optimizer", "hparam", "dropout", "clipping", "amp", "init", "final", "other"]
def _key(e):
    g = RESULTS[e]["cfg"].get("group", "other")
    return (GROUP_ORDER.index(g) if g in GROUP_ORDER else 99, e)

SEED_IDS = [f"base-s{s}" for s in BASE_SEEDS]
FINAL_IDS = [f"final-s{s}" for s in FINAL_SEEDS]
EVAL_FOR = {"base-s1": EVAL_BASE, "final-s1": EVAL_FINAL}   # eval CHỈ cho baseline và cấu hình cuối

rows = []
for e in sorted(RESULTS, key=_key):
    rows.append(to_row(RESULTS[e], eval_scores=EVAL_FOR.get(e), notes=NOTES.get(e, "")))

def _f1(e):
    return RESULTS[e]["summary"]["val_macro_f1"] if e in RESULTS else float("nan")

SUMMARY_NOTES = {
    "baseline": f"{BASE['optimizer']} lr={BASE_LR:g}, {EPOCHS} ep. TB val macro-F1 = "
                f"{NOISE['val_macro_f1']['mean']:.4f} ± {NOISE['val_macro_f1']['std']:.4f} "
                f"(2σ = {NOISE['val_macro_f1']['two_sigma']:.4f}, 3 seed). Mốc đoán đa số = "
                f"{data['majority_acc_val']:.4f}.",
    "loss": f"MSE cho val macro-F1 = {_f1('loss-mse'):.4f} so với CE = {_f1('base-s1'):.4f}; "
            f"loss khác thang đo nên chỉ so bằng macro-F1.",
    "optimizer": f"Bộ tối ưu tốt nhất theo val: {BEST_OPT_CFG['optimizer']} lr={BEST_OPT_CFG['lr']:g} "
                 f"(val macro-F1 {RESULTS[BEST_OPT_EID]['summary']['val_macro_f1']:.4f}). "
                 "Mỗi bộ thử 3 lr; SGD nhạy lr nhất.",
    "hparam": "Batch, kiến trúc, weight decay, cosine, số epoch — xem notes từng dòng.",
    "dropout": f"q chọn cho cấu hình cuối: {Q_DROP:g}.",
    "clipping": f"c = {CLIP_C:g}; thí nghiệm phản chứng ở lr = {HIGH_LR:g}.",
    "amp": "FP16/BF16 không nhanh hơn FP32 trên mạng nhỏ này; xem time/epoch và peak mem.",
    "init": "zeros hỏng vì mất đối xứng; he giữ phương sai kích hoạt qua tầng ReLU.",
    "final": f"Ghép từ val; eval macro-F1 = {EVAL_FINAL['macro_f1']:.4f} "
             f"(cải thiện {EVAL_FINAL['macro_f1'] - EVAL_BASE['macro_f1']:+.4f} so với baseline).",
}
write_xlsx(rows, os.path.join(REPO_ROOT, "templates", "experiment_table_template.xlsx"),
           f"{OUT_DIR}/experiments.xlsx", seed_ids=SEED_IDS, summary_notes=SUMMARY_NOTES)

# ---- kiểm tra: mỗi dòng bảng phải có đúng một ảnh figures/<exp_id>.png ----
png = {f[:-4] for f in os.listdir(FIGS_DIR) if f.endswith(".png")}
missing = sorted(e for e in RESULTS if e not in png)
extra = sorted(png - set(RESULTS))
print(f"\nSố dòng thí nghiệm trong bảng : {len(rows)}")
print(f"Số ảnh figures/<exp_id>.png    : {len(png & set(RESULTS))}")
print(f"Thiếu ảnh                     : {missing or 'không có'}")
print(f"Ảnh khác (không phải exp_id)  : {extra or 'không có'}")
assert not missing, f"thiếu ảnh cho: {missing}"

In [ ]:
# ---- BẢNG SỐ LIỆU CHO BÁO CÁO (chép từ đây) ----
print("=" * 78); print("BẢNG SỐ LIỆU CHO REPORT.md"); print("=" * 78)
print(f"[2] loss bước 0 (He)          = {res_base['summary']['step0_loss']:.4f}   (ln 7 = {LN_C:.4f})")
print(f"[2] quá khớp 20 mẫu, loss cuối = {losses[-1]:.2e}")
print(f"[2] baseline val macro-F1     = {NOISE['val_macro_f1']['mean']:.4f} ± {NOISE['val_macro_f1']['std']:.4f} (σ)")
print(f"[2] baseline val accuracy     = {NOISE['val_acc']['mean']:.4f} ± {NOISE['val_acc']['std']:.4f} (σ)")
print(f"[2] NGƯỠNG NHIỄU 2σ           = {NOISE['val_macro_f1']['two_sigma']:.4f}  (val macro-F1)")
print(f"[2] mốc 'đoán lớp đa số'      = {data['majority_acc_val']:.4f} accuracy")
print(f"[4] eval baseline   : acc {EVAL_BASE['accuracy']:.4f}  macro-F1 {EVAL_BASE['macro_f1']:.4f}")
print(f"[4] eval cấu hình cuối: acc {EVAL_FINAL['accuracy']:.4f}  macro-F1 {EVAL_FINAL['macro_f1']:.4f}")
print(f"[4] val vs eval (cấu hình cuối): {res_final['summary']['val_macro_f1']:.4f} vs {EVAL_FINAL['macro_f1']:.4f}")
print("\nBảng đầy đủ (mỗi thí nghiệm một dòng):")
display(pd.DataFrame(rows)[["exp_id", "group", "val_macro_f1", "val_acc", "best_val_loss",
                            "best_epoch", "time_per_epoch_s", "peak_mem_MB", "diverged",
                            "eval_macro_f1", "eval_acc"]])
print("\nCác file đã sinh trong", OUT_DIR)
for d in ("figures", "results"):
    fs = sorted(os.listdir(f"{OUT_DIR}/{d}"))
    print(f"  {d}/ ({len(fs)} file)")
print("  " + ", ".join(x for x in ("experiments.xlsx", "predictions_eval.csv", "eval_result.json") if os.path.isfile(f"{OUT_DIR}/{x}")))

---
## 5. Đóng gói thư mục nộp

Tạo `submission_<MSSV>/` đúng cấu trúc README mục 6.1, rồi nén lại để nộp.
Đặt tên thư mục bằng cách sửa `LAB_SUBMISSION` hoặc dòng `SUBMISSION_NAME` bên dưới.

In [ ]:
import shutil

_def_name = os.environ.get("LAB_SUBMISSION")
if not _def_name:                       # nếu OUT_DIR đã là thư mục nộp thì dùng luôn tên đó
    _b = os.path.basename(os.path.abspath(OUT_DIR))
    _def_name = _b if _b.startswith("submission") else "submission"
SUBMISSION_NAME = _def_name
PKG = os.path.join(REPO_ROOT, SUBMISSION_NAME)

def _same(a, b):
    return os.path.abspath(a) == os.path.abspath(b)

# Khi nào CẦN gói? Khi PKG khác OUT_DIR — kể cả PKG nằm sâu hơn bên trong OUT_DIR.
# Trường hợp Kaggle: kết quả ở /kaggle/working, gói vào /kaggle/working/repo/submission.
# (Cách cũ dùng _inside() nên coi "PKG nằm trong OUT_DIR" là không cần gói -> nhảy ảnh = 0.)
NEED_PACK = not _same(OUT_DIR, PKG)

if NEED_PACK:
    os.makedirs(PKG, exist_ok=True)
    # figures/ và results/
    for name in ("figures", "results"):
        src, dst = os.path.join(OUT_DIR, name), os.path.join(PKG, name)
        if os.path.isdir(src):
            shutil.rmtree(dst, ignore_errors=True)
            shutil.copytree(src, dst)
    # file phẳng
    for f in ("experiments.xlsx", "predictions_eval.csv", "eval_result.json",
              "eval_result_baseline.json", "REPORT.md"):
        src = os.path.join(OUT_DIR, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(PKG, f))
    # toàn bộ code vào PKG/code/  (README 6.3: code chỉ được nằm trong code/)
    code_dst = os.path.join(PKG, "code")
    os.makedirs(code_dst, exist_ok=True)
    # code: lấy .py / .txt từ CODE_DIR
    for f in sorted(os.listdir(CODE_DIR)):
        if f.endswith((".py", ".txt")) and not f.startswith(("_", ".")):
            shutil.copy2(os.path.join(CODE_DIR, f), os.path.join(code_dst, f))
    # notebook phải là bản ĐÃ CÓ OUTPUT — ưu tiên bản nằm ở CWD (chính notebook đang chạy)
    nb_src = None
    for d in (CWD, CODE_DIR):
        cands = sorted(f for f in os.listdir(d)
                       if f.endswith(".ipynb") and not f.startswith(("_", ".")))
        if cands:
            nb_src = os.path.join(d, cands[0])
            break
    if nb_src:
        shutil.copy2(nb_src, os.path.join(code_dst, "lab.ipynb"))
        print(f"notebook nộp: {nb_src}")
    else:
        print("CẢNH BÁO: không tìm thấy .ipynb để đóng gói — hãy chép lab.ipynb vào thư mục nộp.")
    print(f"đã đóng gói vào {PKG}")
else:
    print(f"KHÔNG cần đóng gói: OUT_DIR đã là thư mục nộp ({OUT_DIR})")

print("\nCấu trúc thư mục nộp:")
for root, dirs, files in os.walk(PKG):
    dirs[:] = [d for d in dirs if d not in ("__pycache__", ".ipynb_checkpoints")]
    rel = os.path.relpath(root, PKG)
    depth = 0 if rel == "." else rel.count(os.sep) + 1
    if depth <= 1:
        print("   " * depth + (os.path.basename(root) + "/" if rel != "." else "./"))
        if depth == 1:
            for fn in sorted(files)[:8]:
                print("   " * depth + f"  {fn}")
            if len(files) > 8:
                print("   " * depth + f"  ... ({len(files)} file)")

fg = os.path.join(PKG, "figures")
allpng = [f for f in os.listdir(fg) if f.endswith(".png")] if os.path.isdir(fg) else []
cmp_png = [f for f in allpng if f.startswith("compare_")]
exp_ids = [r["exp_id"] for r in rows]
row_png = [f for f in allpng if f[:-4] in set(exp_ids)]
print(f"\nfigures/: {len(allpng)} ảnh = {len(row_png)} ảnh <exp_id>.png "
      f"(đúng bằng {len(rows)} dòng bảng) + {len(cmp_png)} ảnh compare_* "
      f"+ {len(allpng)-len(row_png)-len(cmp_png)} ảnh khác")
missing = [e for e in exp_ids if not os.path.isfile(os.path.join(fg, f"{e}.png"))]
print("Thiếu ảnh:", missing or "không có")
assert not missing, f"thiếu ảnh cho: {missing}"
print("KHÔNG nộp: data/, *.npz, *.pt, __pycache__/, .ipynb_checkpoints/ (xem README mục 6.3).")